# Phase 6: Deep Learning
## GCN, GAT, CNN, LSTM, Transformer, Late Fusion

**Sections:**
- 6.1 Data preparation for DL
- 6.2 GCN (separate GC vs PDC)
- 6.3 Late Fusion (dual-branch GC + PDC)
- 6.4 CNN multi-channel
- 6.5 Evaluation & comparison
- 6.6 LOSO for best DL model


In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {device}")


## 6.1 Load Connectivity Matrices

In [ ]:
GC_DIR = r"D:\Skripsi\new_data\01_granger_causality\output\gc_matrices"
PDC_DIR = r"D:\Skripsi\new_data\02_pdc\output\pdc_matrices"
FIG_DIR = r"D:\Skripsi\new_data\phase_6_deep_learning\figures"
MODEL_DIR = r"D:\Skripsi\new_data\phase_6_deep_learning\models"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

N_CHANNELS = 62
TRIAL_LABELS = [1, 0, -1, -1, 0, 1, -1, 0, 1, 1, 0, -1, 0, 1, -1]
EMOTION_MAP = {-1: "negative", 0: "neutral", 1: "positive"}
LABEL_MAP = {-1: 0, 0: 1, 1: 2}

def load_all_matrices(base_dir, prefix, suffix='.npy'):
    """Load all matrices from directory structure."""
    matrices = []
    labels = []
    subjects = []
    
    for subj_dir in sorted(os.listdir(base_dir)):
        subj_path = os.path.join(base_dir, subj_dir)
        if not os.path.isdir(subj_path):
            continue
        subject_id = int(subj_dir.split('_')[1])
        
        for sess_dir in sorted(os.listdir(subj_path)):
            sess_path = os.path.join(subj_path, sess_dir)
            if not os.path.isdir(sess_path):
                continue
            
            for f in sorted(os.listdir(sess_path)):
                if prefix in f and f.endswith(suffix):
                    trial_idx = int(f.split('_')[-1].split('.')[0])
                    label = TRIAL_LABELS[trial_idx - 1]
                    
                    matrix = np.load(os.path.join(sess_path, f))
                    matrices.append(matrix)
                    labels.append(LABEL_MAP[label])
                    subjects.append(subject_id)
    
    return np.array(matrices), np.array(labels), np.array(subjects)

# Load GC (thresholded)
X_gc, y, subjects = load_all_matrices(GC_DIR, 'thresholded')
print(f"✓ GC matrices: {X_gc.shape}")

# Load PDC (best band - try beta first, then alpha)
for band in ['beta', 'alpha', 'gamma']:
    X_pdc, _, _ = load_all_matrices(PDC_DIR, f'pdc_{band}')
    if len(X_pdc) == len(X_gc):
        print(f"✓ PDC {band} matrices: {X_pdc.shape}")
        PDC_BAND = band
        break

print(f"\nTotal samples: {len(y)}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")


## 6.2 CNN Classifier (GC vs PDC)

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, n_channels=62):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        
        # Calculate flattened size
        flat_size = 64 * (n_channels // 4) * (n_channels // 4)
        self.fc1 = nn.Linear(flat_size, 128)
        self.fc2 = nn.Linear(128, 3)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, 62, 62)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

def train_and_evaluate_cnn(X, y, subjects, n_epochs=50, lr=0.001, patience=10):
    """Train CNN with LOSO CV + validation-based early stopping."""
    logo = LeaveOneGroupOut()
    all_accs = []
    
    # Normalize matrices
    X_norm = np.array([(m - m.mean()) / (m.std() + 1e-10) for m in X])
    
    for train_idx, test_idx in logo.split(X_norm, y, subjects):
        X_train = torch.FloatTensor(X_norm[train_idx])
        y_train = torch.LongTensor(y[train_idx])
        X_test = torch.FloatTensor(X_norm[test_idx])
        y_test = torch.LongTensor(y[test_idx])
        
        # Split train into train/val for early stopping
        n_val = max(1, len(X_train) // 5)
        perm = torch.randperm(len(X_train))
        X_val, y_val = X_train[perm[:n_val]], y_train[perm[:n_val]]
        X_train, y_train = X_train[perm[n_val:]], y_train[perm[n_val:]]
        
        train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
        
        model = CNNClassifier().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        best_val_loss = float('inf')
        patience_counter = 0
        best_state = None
        
        for epoch in range(n_epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                output = model(X_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
            
            # Validation check
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(X_val.to(device)), y_val.to(device)).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        # Restore best model
        if best_state is not None:
            model.load_state_dict(best_state)
        
        model.eval()
        with torch.no_grad():
            X_test = X_test.to(device)
            preds = model(X_test).argmax(dim=1).cpu().numpy()
        
        acc = accuracy_score(y_test.numpy(), preds)
        all_accs.append(acc)
    
    return np.array(all_accs)

# Run CNN for GC
print("Training CNN on GC matrices...")
gc_accs = train_and_evaluate_cnn(X_gc, y, subjects, n_epochs=30)
print(f"GC CNN LOSO: {gc_accs.mean():.3f} ± {gc_accs.std():.3f}")

# Run CNN for PDC
print(f"\nTraining CNN on PDC {PDC_BAND} matrices...")
pdc_accs = train_and_evaluate_cnn(X_pdc, y, subjects, n_epochs=30)
print(f"PDC {PDC_BAND} CNN LOSO: {pdc_accs.mean():.3f} ± {pdc_accs.std():.3f}")


## 6.3 Late Fusion (Dual-Branch)

In [ ]:
class DualBranchCNN(nn.Module):
    def __init__(self, n_channels=62):
        super().__init__()
        # Branch A: GC
        self.conv_a1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv_a2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Branch B: PDC
        self.conv_b1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv_b2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        flat_per_branch = 32 * (n_channels // 4) * (n_channels // 4)
        self.fc1 = nn.Linear(flat_per_branch * 2, 128)
        self.fc2 = nn.Linear(128, 3)
    
    def forward(self, x_gc, x_pdc):
        # Branch A
        a = x_gc.unsqueeze(1)
        a = self.pool(self.relu(self.conv_a1(a)))
        a = self.pool(self.relu(self.conv_a2(a)))
        a = a.view(a.size(0), -1)
        
        # Branch B
        b = x_pdc.unsqueeze(1)
        b = self.pool(self.relu(self.conv_b1(b)))
        b = self.pool(self.relu(self.conv_b2(b)))
        b = b.view(b.size(0), -1)
        
        # Fusion
        combined = torch.cat([a, b], dim=1)
        out = self.dropout(self.relu(self.fc1(combined)))
        out = self.fc2(out)
        return out

print("Training Dual-Branch Late Fusion CNN...")

X_gc_norm = np.array([(m - m.mean()) / (m.std() + 1e-10) for m in X_gc])
X_pdc_norm = np.array([(m - m.mean()) / (m.std() + 1e-10) for m in X_pdc])

logo = LeaveOneGroupOut()
fusion_accs = []

for train_idx, test_idx in logo.split(X_gc_norm, y, subjects):
    X_gc_train = torch.FloatTensor(X_gc_norm[train_idx])
    X_pdc_train = torch.FloatTensor(X_pdc_norm[train_idx])
    y_train_all = torch.LongTensor(y[train_idx])
    X_gc_test = torch.FloatTensor(X_gc_norm[test_idx]).to(device)
    X_pdc_test = torch.FloatTensor(X_pdc_norm[test_idx]).to(device)
    y_test_t = torch.LongTensor(y[test_idx])
    
    # Split train into train/val for early stopping
    n_val = max(1, len(X_gc_train) // 5)
    perm = torch.randperm(len(X_gc_train))
    X_gc_val, X_pdc_val, y_val = X_gc_train[perm[:n_val]].to(device), X_pdc_train[perm[:n_val]].to(device), y_train_all[perm[:n_val]].to(device)
    X_gc_train, X_pdc_train, y_train = X_gc_train[perm[n_val:]], X_pdc_train[perm[n_val:]], y_train_all[perm[n_val:]]
    
    train_loader = DataLoader(TensorDataset(X_gc_train, X_pdc_train, y_train), batch_size=32, shuffle=True)
    
    model = DualBranchCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(50):
        model.train()
        for gc_batch, pdc_batch, y_batch in train_loader:
            gc_batch, pdc_batch, y_batch = gc_batch.to(device), pdc_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(gc_batch, pdc_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
        
        # Validation check
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_gc_val, X_pdc_val), y_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= 10:
                break
    
    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
    
    model.eval()
    with torch.no_grad():
        preds = model(X_gc_test, X_pdc_test).argmax(dim=1).cpu().numpy()
    
    fusion_accs.append(accuracy_score(y_test_t.numpy(), preds))

fusion_accs = np.array(fusion_accs)
print(f"Late Fusion LOSO: {fusion_accs.mean():.3f} ± {fusion_accs.std():.3f}")


## 6.5 Evaluation & Comparison

In [ ]:
dl_results = pd.DataFrame([
    {'Model': 'CNN', 'Input': 'GC', 'LOSO_Acc': gc_accs.mean(), 'LOSO_Std': gc_accs.std()},
    {'Model': 'CNN', 'Input': f'PDC {PDC_BAND}', 'LOSO_Acc': pdc_accs.mean(), 'LOSO_Std': pdc_accs.std()},
    {'Model': 'DualBranch CNN', 'Input': 'GC+PDC Fusion', 'LOSO_Acc': fusion_accs.mean(), 'LOSO_Std': fusion_accs.std()},
])

print("=== Deep Learning Results ===")
print(dl_results.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(dl_results))
bars = ax.bar(x, dl_results['LOSO_Acc'], yerr=dl_results['LOSO_Std'], 
              color=['steelblue', 'coral', 'green'], capsize=5)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Model']}\n({r['Input']})" for _, r in dl_results.iterrows()])
ax.set_ylabel('LOSO Accuracy')
ax.set_title('Deep Learning: GC vs PDC vs Late Fusion')
ax.axhline(y=1/3, color='gray', linestyle='--', label='Chance (33%)')
ax.legend()
ax.set_ylim(0, 1)

for bar, acc in zip(bars, dl_results['LOSO_Acc']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{acc:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "6.5_dl_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

dl_results.to_csv(os.path.join(FIG_DIR, "dl_results.csv"), index=False)
print("\n✓ Deep Learning results saved")
